In [3]:
import pandas as pd

df_freq = pd.read_csv("../../data/freMTPL2freq.csv")
df_sev = pd.read_csv("../../data/df_sev_clean.csv")

# ۱. ادغام داده‌های خسارت با داده‌های بیمه‌نامه
claims_per_policy = df_sev.groupby('IDpol')['ClaimAmount'].sum().reset_index()
df_merged = pd.merge(df_freq, claims_per_policy, on='IDpol', how='left')
df_merged['ClaimAmount'] = df_merged['ClaimAmount'].fillna(0)

# ۲. گروه‌بندی بر اساس منطقه (Region) و محاسبه KPIها
region_summary = df_merged.groupby('Region').agg(
    Total_Exposure=('Exposure', 'sum'),
    Total_Claims=('ClaimNb', 'sum'),
    Total_Claim_Cost=('ClaimAmount', 'sum')
).reset_index()

# ۳. محاسبه Claim Frequency و Average Severity برای هر منطقه
region_summary['Claim_Frequency'] = region_summary['Total_Claims'] / region_summary['Total_Exposure']
region_summary['Average_Severity'] = region_summary['Total_Claim_Cost'] / region_summary['Total_Claims']

# جایگزینی NaN با ۰ در صورت عدم وجود ادعا
region_summary['Average_Severity'] = region_summary['Average_Severity'].fillna(0)

# مرتب‌سازی بر اساس مجموع هزینه خسارت (Total Claim Cost) برای شناسایی مناطق پربار
region_summary = region_summary.sort_values(by='Total_Claim_Cost', ascending=False)

# منطقه Champagne-Ardenne شایسته توجه است زیرا بالاترین میانگین شدت خسارت
# (حدود ۶۲۵۰ به ازای هر ادعا) را دارد؛ یعنی خسارت‌ها در این منطقه
# به‌طور غیرمعمولی گران هستند.
#
# اگرچه فراوانی ادعاها در این منطقه تنها حدود ۶.۴٪ است،
# اما شدت بالای خسارت نشان می‌دهد که خسارت‌ها در صورت وقوع
# می‌توانند از نظر مالی قابل توجه باشند.
#
# بنابراین، این منطقه باید بر اساس شدت خسارت مورد بررسی قرار گیرد،
# نه صرفاً بر اساس تعداد کل ادعاها.

print(region_summary[['Region', 'Total_Exposure', 'Claim_Frequency', 'Total_Claim_Cost', 'Average_Severity']])

                         Region  Total_Exposure  Claim_Frequency  \
6                        Centre   102707.853386         0.063043   
21                  Rhone-Alpes    45343.547286         0.093354   
20  Provence-Alpes-Cotes-D'Azur    35790.321015         0.083430   
11                Ile-de-France    30207.161954         0.085774   
5                      Bretagne    27752.443675         0.067417   
17             Pays-de-la-Loire    21931.151165         0.071861   
12         Languedoc-Roussillon    14736.104177         0.072475   
1                     Aquitaine    14322.166692         0.073662   
16           Nord-Pas-de-Calais    11496.974269         0.082109   
19             Poitou-Charentes    11163.408715         0.071663   
14                     Lorraine     8113.777972         0.057680   
3               Basse-Normandie     6657.801142         0.067890   
18                     Picardie     3574.410221         0.087847   
4                     Bourgogne     5025.098609 